# Phase 2 — Data Preprocessing

This phase cleans source strings on a copy of the original dataset, preserves rows and missing-value indicators, and derives price features only when both observed prices are available. Missing review text or optional media does not remove a product.

In [2]:

import pandas as pd
import numpy as np
import re

# ============================================================
# 1. LOAD ORIGINAL DATASET
# ============================================================

df_original = pd.read_csv("../backend/electronics.csv")
df_clean = df_original.copy()

print("Original dataset shape:", df_original.shape)


# ============================================================
# 2. CLEAN PRICE COLUMNS
# ============================================================

def clean_price(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = re.sub(r"[^\d.]", "", value)

    try:
        return float(value) if value else np.nan
    except ValueError:
        return np.nan


df_clean["discount_price"] = df_clean["discount_price"].apply(clean_price)
df_clean["actual_price"] = df_clean["actual_price"].apply(clean_price)


# ============================================================
# 3. CLEAN REVIEW RATINGS
# ============================================================

def clean_rating(value):
    if pd.isna(value):
        return np.nan

    match = re.search(r"\d+(?:\.\d+)?", str(value))

    if match:
        rating = float(match.group())

        if 0 <= rating <= 5:
            return rating

    return np.nan


df_clean["review_rating"] = df_clean["review_rating"].apply(clean_rating)


# ============================================================
# 4. CLEAN REVIEW COUNTS
# ============================================================

def clean_review_count(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower().replace(",", "")

    # Extract numeric value and optional K/M suffix
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*([km]?)",
        value
    )

    if not match:
        return np.nan

    number = float(match.group(1))
    suffix = match.group(2)

    if suffix == "k":
        number *= 1000
    elif suffix == "m":
        number *= 1000000

    return int(number)


df_clean["no_of_ratings"] = (
    df_clean["no_of_ratings"].apply(clean_review_count)
)


# ============================================================
# 5. CLEAN PRODUCT TITLES
# ============================================================

def clean_title(value):
    if pd.isna(value):
        return np.nan

    value = str(value).lower()
    value = re.sub(r"\s+", " ", value)

    return value.strip()


df_clean["name"] = df_clean["name"].apply(clean_title)


# ============================================================
# 6. STANDARDIZE CATEGORY TEXT
# ============================================================

for column in ["main_category", "sub_category"]:
    df_clean[column] = df_clean[column].apply(
        lambda x: str(x).strip().lower()
        if pd.notna(x) else np.nan
    )


# ============================================================
# 7. ENSURE NUMERIC DATA TYPES
# ============================================================

numeric_columns = [
    "discount_price",
    "actual_price",
    "review_rating",
    "no_of_ratings"
]

for column in numeric_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )


# ============================================================
# 8. PREPROCESSING SUMMARY
# ============================================================

print("\nUpdated data types:")
print(df_clean.dtypes)

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nNumerical statistics:")
display(df_clean[numeric_columns].describe())


# ============================================================
# 9. DATA VALIDATION
# ============================================================

print("\nDATA VALIDATION")

print(
    "Invalid discount prices:",
    (df_clean["discount_price"] <= 0).sum()
)

print(
    "Invalid actual prices:",
    (df_clean["actual_price"] <= 0).sum()
)

print(
    "Discount price greater than actual price:",
    (
        df_clean["discount_price"] >
        df_clean["actual_price"]
    ).sum()
)

print(
    "Invalid ratings:",
    (
        (df_clean["review_rating"] < 0) |
        (df_clean["review_rating"] > 5)
    ).sum()
)

print(
    "Missing review counts:",
    df_clean["no_of_ratings"].isna().sum()
)


# ============================================================
# 10. SAVE INTERMEDIATE DATASET
# ============================================================

output_path = "../electronics_preprocessed_stage1.csv"

df_clean.to_csv(output_path, index=False)

print("\nPreprocessing Stage 1 completed!")
print("Final shape:", df_clean.shape)
print("Saved to:", output_path)


Original dataset shape: (5010, 10)

Updated data types:
name               object
main_category      object
sub_category       object
image              object
link               object
no_of_ratings     float64
discount_price    float64
actual_price      float64
review_rating     float64
review_text        object
dtype: object

Missing values:
name                0
main_category       0
sub_category        0
image               0
link                0
no_of_ratings      52
discount_price    270
actual_price       31
review_rating       0
review_text         3
dtype: int64

Numerical statistics:
       discount_price   actual_price  review_rating  no_of_ratings
count     4740.000000    4979.000000    5010.000000    4958.000000
mean      3078.190819    5226.881996       4.488423   13260.089351
std       8564.866249   11861.096022       1.086022   34803.225962
min         36.000000      35.000000       1.000000       1.000000
25%        299.000000     835.000000       4.000000     509.25

In [3]:

import pandas as pd
import numpy as np

# ============================================================
# STAGE 2: MISSING VALUE HANDLING AND FEATURE CREATION
# ============================================================

# Load Stage 1 output
df_clean = pd.read_csv("../electronics_preprocessed_stage1.csv")

print("Starting shape:", df_clean.shape)


# ============================================================
# 1. CREATE MISSING VALUE INDICATORS
# ============================================================

# These columns indicate whether the original value was missing.

df_clean["discount_price_missing"] = (
    df_clean["discount_price"].isna().astype(int)
)

df_clean["actual_price_missing"] = (
    df_clean["actual_price"].isna().astype(int)
)

df_clean["review_count_missing"] = (
    df_clean["no_of_ratings"].isna().astype(int)
)

df_clean["review_text_missing"] = (
    df_clean["review_text"].isna().astype(int)
)


# ============================================================
# 2. CHECK PRICE AVAILABILITY
# ============================================================

# A product can be used for price-based calculations
# only when both prices are available and valid.

df_clean["price_available"] = (
    df_clean["discount_price"].notna() &
    df_clean["actual_price"].notna() &
    (df_clean["discount_price"] > 0) &
    (df_clean["actual_price"] > 0)
).astype(int)


# ============================================================
# 3. CREATE DISCOUNT AMOUNT
# ============================================================

df_clean["discount_amount"] = np.where(
    df_clean["price_available"] == 1,
    df_clean["actual_price"] - df_clean["discount_price"],
    np.nan
)


# ============================================================
# 4. CREATE DISCOUNT PERCENTAGE
# ============================================================

df_clean["discount_percent"] = np.where(
    df_clean["price_available"] == 1,
    (
        df_clean["discount_amount"] /
        df_clean["actual_price"]
    ) * 100,
    np.nan
)


# ============================================================
# 5. VALIDATE DISCOUNT FEATURES
# ============================================================

print("\nPRICE FEATURE VALIDATION")

print(
    "Products with both prices available:",
    df_clean["price_available"].sum()
)

print(
    "Products without both prices:",
    (df_clean["price_available"] == 0).sum()
)

print(
    "Missing discount amounts:",
    df_clean["discount_amount"].isna().sum()
)

print(
    "Missing discount percentages:",
    df_clean["discount_percent"].isna().sum()
)

print(
    "Negative discount amounts:",
    (df_clean["discount_amount"] < 0).sum()
)


# ============================================================
# 6. INSPECT NEW FEATURES
# ============================================================

print("\nNEW PRICE FEATURES")

display(
    df_clean[
        [
            "name",
            "actual_price",
            "discount_price",
            "discount_amount",
            "discount_percent",
            "price_available"
        ]
    ].head(15)
)


# ============================================================
# 7. MISSING VALUE SUMMARY
# ============================================================

print("\nUPDATED MISSING VALUE SUMMARY")

missing_summary = pd.DataFrame({
    "Missing Count": df_clean.isnull().sum(),
    "Missing Percentage": (
        df_clean.isnull().sum() / len(df_clean) * 100
    ).round(2)
})

display(missing_summary)


# ============================================================
# 8. SAVE STAGE 2 DATASET
# ============================================================

output_path = "../electronics_preprocessed_stage2.csv"

df_clean.to_csv(output_path, index=False)

print("\n" + "=" * 60)
print("STAGE 2 COMPLETED")
print("=" * 60)

print("Final shape:", df_clean.shape)
print("Saved:", output_path)

print("Original dataset remains unchanged.")
print("Missing values have not been artificially imputed.")


Starting shape: (5010, 10)

PRICE FEATURE VALIDATION
Products with both prices available: 4740
Products without both prices: 270
Missing discount amounts: 270
Missing discount percentages: 270
Negative discount amounts: 0

NEW PRICE FEATURES
                                                                                                                             name  actual_price  discount_price  discount_amount  discount_percent  price_available
0                                                                            redmi 10 power (power black, 8gb ram, 128gb storage)       18999.0         10999.0           8000.0         42.107479                1
1                                                                   oneplus nord ce 2 lite 5g (blue tide, 6gb ram, 128gb storage)       19999.0         18999.0           1000.0          5.000250                1
2   oneplus bullets z2 bluetooth wireless in ear earphones with mic, bombastic bass - 12.4 mm drivers, 10 mins charge - 20

In [4]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# STAGE 3: OUTLIER DETECTION AND INVESTIGATION
# ============================================================

df_clean = pd.read_csv("../electronics_preprocessed_stage2.csv")

print("Dataset shape:", df_clean.shape)


# ============================================================
# 1. SELECT NUMERICAL FEATURES
# ============================================================

columns_to_check = [
    "discount_price",
    "actual_price",
    "no_of_ratings",
    "discount_percent"
]


# ============================================================
# 2. CALCULATE IQR AND OUTLIER LIMITS
# ============================================================

outlier_results = []

for column in columns_to_check:

    data = df_clean[column].dropna()

    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = data[
        (data < lower_limit) |
        (data > upper_limit)
    ]

    outlier_results.append({
        "Feature": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Limit": lower_limit,
        "Upper Limit": upper_limit,
        "Outlier Count": len(outliers),
        "Outlier Percentage": round(
            len(outliers) / len(data) * 100, 2
        )
    })


outlier_summary = pd.DataFrame(outlier_results)

print("\n" + "=" * 65)
print("IQR OUTLIER SUMMARY")
print("=" * 65)

display(outlier_summary)


# ============================================================
# 3. DISPLAY EXTREME PRICE RECORDS
# ============================================================

print("\nHIGHEST ACTUAL PRICE PRODUCTS")

display(
    df_clean.nlargest(10, "actual_price")[
        ["name", "actual_price", "discount_price",
         "review_rating", "no_of_ratings"]
    ]
)

print("\nHIGHEST DISCOUNT PRICE PRODUCTS")

display(
    df_clean.nlargest(10, "discount_price")[
        ["name", "discount_price", "actual_price",
         "review_rating"]
    ]
)


# ============================================================
# 4. DISPLAY PRODUCTS WITH HIGHEST REVIEW COUNTS
# ============================================================

print("\nPRODUCTS WITH HIGHEST REVIEW COUNTS")

display(
    df_clean.nlargest(10, "no_of_ratings")[
        ["name", "no_of_ratings", "review_rating"]
    ]
)


# ============================================================
# 5. DISPLAY HIGHEST DISCOUNT PERCENTAGES
# ============================================================

print("\nPRODUCTS WITH HIGHEST DISCOUNT PERCENTAGES")

display(
    df_clean.nlargest(10, "discount_percent")[
        ["name", "actual_price", "discount_price",
         "discount_percent"]
    ]
)


# ============================================================
# 6. VISUALIZE OUTLIERS USING BOXPLOTS
# ============================================================

for column in columns_to_check:

    plt.figure(figsize=(9, 4))

    plt.boxplot(
        df_clean[column].dropna(),
        vert=False
    )

    plt.title("Outlier Detection: " + column)
    plt.xlabel(column)

    plt.tight_layout()
    plt.show()


# ============================================================
# 7. CHECK NEGATIVE AND IMPOSSIBLE VALUES
# ============================================================

print("\nDATA VALIDATION")

print(
    "Zero or negative discount prices:",
    (df_clean["discount_price"] <= 0).sum()
)

print(
    "Zero or negative actual prices:",
    (df_clean["actual_price"] <= 0).sum()
)

print(
    "Negative review counts:",
    (df_clean["no_of_ratings"] < 0).sum()
)

print(
    "Discount percentages below 0:",
    (df_clean["discount_percent"] < 0).sum()
)

print(
    "Discount percentages above 100:",
    (df_clean["discount_percent"] > 100).sum()
)


# ============================================================
# 8. SAVE OUTLIER ANALYSIS
# ============================================================

outlier_summary.to_csv(
    "../outlier_analysis.csv",
    index=False
)

print("\n" + "=" * 65)
print("STAGE 3 OUTLIER INVESTIGATION COMPLETED")
print("=" * 65)

print("No records have been deleted or modified.")
print("Dataset shape:", df_clean.shape)
print("Outlier report saved as outlier_analysis.csv")


Dataset shape: (5010, 17)

IQR OUTLIER SUMMARY
            Feature          Q1           Q3          IQR   Lower Limit  Upper Limit  Outlier Count  Outlier Percentage
0    discount_price  299.000000  1699.000000  1400.000000  -1801.000000    3799.0000            633               13.35
1      actual_price  835.000000  3999.000000  3164.000000  -3911.000000    8745.0000            632               12.69
2     no_of_ratings  509.250000  9812.750000  9303.500000 -13446.000000   23768.0000            681               13.74
3  discount_percent   35.035035    70.023341    34.988306    -17.447424     122.5058              0                0.00

HIGHEST ACTUAL PRICE PRODUCTS
                                                                                                                               name  actual_price  discount_price  review_rating  no_of_ratings
1682                                                                       samsung galaxy s23 ultra 5g (green, 12gb, 256gb storage

In [5]:

import pandas as pd
import numpy as np

# ============================================================
# STAGE 4: REPEATED LISTING INVESTIGATION
# ============================================================

df_clean = pd.read_csv("../electronics_preprocessed_stage2.csv")

print("Dataset shape:", df_clean.shape)


# ============================================================
# 1. EXACT DUPLICATE ROWS
# ============================================================

exact_duplicates = df_clean.duplicated().sum()

print("\nEXACT DUPLICATE ROWS:", exact_duplicates)


# ============================================================
# 2. DUPLICATE PRODUCT NAMES
# ============================================================

repeated_products = df_clean[
    df_clean.duplicated("name", keep=False)
].copy()

print(
    "\nRows with repeated product names:",
    len(repeated_products)
)

print(
    "Distinct repeated product names:",
    repeated_products["name"].nunique()
)


# ============================================================
# 3. GROUP REPEATED PRODUCTS
# ============================================================

repeated_analysis = (
    df_clean.groupby("name")
    .agg(
        occurrences=("name", "size"),
        unique_links=("link", "nunique"),
        unique_discount_prices=("discount_price", "nunique"),
        unique_actual_prices=("actual_price", "nunique"),
        unique_ratings=("review_rating", "nunique"),
        unique_review_counts=("no_of_ratings", "nunique")
    )
)

repeated_analysis = repeated_analysis[
    repeated_analysis["occurrences"] > 1
].sort_values("occurrences", ascending=False)

print("\nREPEATED PRODUCT ANALYSIS")

display(repeated_analysis.head(20))


# ============================================================
# 4. CHECK WHETHER PRODUCT LINKS ARE REPEATED
# ============================================================

duplicate_links = df_clean[
    df_clean.duplicated("link", keep=False)
].sort_values("link")

print(
    "\nRows belonging to repeated product links:",
    len(duplicate_links)
)

print(
    "Distinct repeated links:",
    duplicate_links["link"].nunique()
)


# ============================================================
# 5. INSPECT PRODUCTS WITH MULTIPLE PRICES
# ============================================================

different_prices = repeated_analysis[
    (repeated_analysis["unique_discount_prices"] > 1) |
    (repeated_analysis["unique_actual_prices"] > 1)
]

print(
    "\nRepeated names with multiple prices:",
    len(different_prices)
)

display(
    different_prices.head(15)
)


# ============================================================
# 6. INSPECT PRODUCTS WITH MULTIPLE RATINGS
# ============================================================

different_ratings = repeated_analysis[
    repeated_analysis["unique_ratings"] > 1
]

print(
    "\nRepeated names with multiple ratings:",
    len(different_ratings)
)

display(
    different_ratings.head(15)
)


# ============================================================
# 7. DISPLAY SAMPLE RECORDS FOR MANUAL INSPECTION
# ============================================================

sample_name = repeated_analysis.index[0]

print("\nSAMPLE PRODUCT:", sample_name)

display(
    df_clean[df_clean["name"] == sample_name][
        [
            "name",
            "link",
            "discount_price",
            "actual_price",
            "review_rating",
            "no_of_ratings"
        ]
    ]
)


# ============================================================
# 8. SAVE INVESTIGATION RESULTS
# ============================================================

repeated_analysis.to_csv(
    "../repeated_products_analysis.csv"
)

print("\n" + "=" * 60)
print("STAGE 4 INVESTIGATION COMPLETED")
print("=" * 60)

print("No records have been deleted.")
print("Analysis saved as repeated_products_analysis.csv")


Dataset shape: (5010, 17)

EXACT DUPLICATE ROWS: 0

Rows with repeated product names: 710
Distinct repeated product names: 300

REPEATED PRODUCT ANALYSIS
                                                                                                                               occurrences  unique_links  unique_discount_prices  unique_actual_prices  unique_ratings  unique_review_counts
name                                                                                                                                                                                                                                        
tagg verve sense smartwatch with 1.70'' large display, real spo2, and real-time heart rate tracking, 7 days battery backu...             7             7                       1                     1               3                     1
fire-boltt visionary 1.78" amoled bluetooth calling smartwatch with 368 * 448 pixel resolution, rotating crown & 60hz refr...          

In [6]:

import pandas as pd
import numpy as np

# ============================================================
# STAGE 5: MISSING VALUE TREATMENT
# ============================================================

df = pd.read_csv("../electronics_preprocessed_stage2.csv")

print("Starting shape:", df.shape)


# 1. Preserve missing-value information

df["discount_price_missing"] = (
    df["discount_price"].isna().astype(int)
)

df["actual_price_missing"] = (
    df["actual_price"].isna().astype(int)
)

df["review_count_missing"] = (
    df["no_of_ratings"].isna().astype(int)
)

df["review_text_missing"] = (
    df["review_text"].isna().astype(int)
)


# 2. Create discount amount only where both prices exist

df["discount_amount"] = (
    df["actual_price"] - df["discount_price"]
).where(df["price_available"] == 1)


# 3. Calculate discount percentage

df["discount_percent"] = (
    df["discount_amount"] /
    df["actual_price"]
) * 100

df["discount_percent"] = df["discount_percent"].where(
    df["price_available"] == 1
)


# 4. Create availability indicator

df["price_available"] = (
    df["discount_price"].notna() &
    df["actual_price"].notna() &
    (df["discount_price"] > 0) &
    (df["actual_price"] > 0)
).astype(int)


# 5. Review count: retain missing values for now
# We will decide imputation separately during ML preparation.


# 6. Review text: preserve missing values
# No artificial review text will be created.


# 7. Generate final missing-value report

missing_report = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (
        df.isna().mean() * 100
    ).round(2)
})

print("\nFINAL MISSING VALUE REPORT")

display(missing_report)


# 8. Verify records were not deleted

print("\nRECORD VALIDATION")
print("Original expected records: 5010")
print("Current records:", len(df))

assert len(df) == 5010


# 9. Save analysis-ready dataset

df.to_csv(
    "../electronics_analysis_ready.csv",
    index=False
)

print("\nSTAGE 5 COMPLETED")
print("Saved: electronics_analysis_ready.csv")
print("Original dataset remains unchanged.")


Starting shape: (5010, 17)

FINAL MISSING VALUE REPORT
                        Missing Count  Missing Percentage
name                                0                0.00
main_category                       0                0.00
sub_category                        0                0.00
image                               0                0.00
link                                0                0.00
no_of_ratings                      52                1.04
discount_price                    270                5.39
actual_price                       31                0.62
review_rating                       0                0.00
review_text                         3                0.06
discount_price_missing              0                0.00
actual_price_missing                0                0.00
review_count_missing                0                0.00
review_text_missing                 0                0.00
price_available                     0                0.00
discount_amount  